# RAG 第 7 课：Generation Evaluation 入门

上一课我们已经学了检索阶段怎么评估，比如：
- `Hit@k`
- `Precision@k`
- `Recall@k`
- `MRR`

但真实 RAG 系统里，仅仅“检索到了相关文档”还不够。
我们还需要继续问：

- 模型最后生成的答案对不对？
- 答案有没有真正用到检索到的上下文？
- 答案是不是在胡编？

这节课我们就来做一个教学版的 `Generation Evaluation`。

In [ ]:
# 先清理 GPU 缓存，保持和前面课程一致
import gc

try:
    import torch
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available, skip GPU cache clear. ({e})')

## 1. 先准备一个小型 RAG 数据集

为了让评估逻辑足够清楚，这里我们继续使用一个很小的知识库。
同时，我们给每个 query 都准备一个参考答案 `reference_answer`，方便做教学版评估。

In [ ]:
import re
from collections import Counter

kb = [
    'E401 means authentication failed because the API token is invalid or expired.',
    'To reset your password, open Settings, then Security, then choose Reset Password.',
    'Chunking splits long documents into smaller passages so retrieval is more focused.',
    'Embeddings map text into vectors so semantic similarity can be measured.',
    'Reranking reorders retrieved candidates so the strongest evidence appears first.',
]

dataset = [
    {
        'query': 'What does E401 mean?',
        'gold_doc_id': 0,
        'reference_answer': 'E401 means authentication failed because the token is invalid or expired.',
    },
    {
        'query': 'How do I reset my password?',
        'gold_doc_id': 1,
        'reference_answer': 'Open Settings, go to Security, and choose Reset Password.',
    },
    {
        'query': 'Why do we use chunking in RAG?',
        'gold_doc_id': 2,
        'reference_answer': 'Chunking breaks long documents into smaller passages so retrieval becomes more focused.',
    },
    {
        'query': 'What are embeddings used for in retrieval?',
        'gold_doc_id': 3,
        'reference_answer': 'Embeddings turn text into vectors so semantic similarity can be measured.',
    },
]

for i, text in enumerate(kb):
    print(f'doc {i}: {text}')

## 2. 一个最小可运行的 RAG pipeline

为了聚焦评估，这里我们使用一个非常小的 pipeline：

1. 用词面匹配做检索
2. 取 top-1 文档作为上下文
3. 用一个教学版生成器，根据上下文产出答案

真实系统当然会更复杂，但评估思路是共通的。

In [ ]:
def normalize_token(token):
    token = token.lower()
    token = re.sub(r'[^a-z0-9]+', '', token)
    if token.endswith('ing') and len(token) > 5:
        token = token[:-3]
    elif token.endswith('s') and len(token) > 4:
        token = token[:-1]
    return token


def tokenize(text):
    return [t for t in (normalize_token(x) for x in text.split()) if t]


def lexical_score(query, text):
    q_tokens = tokenize(query)
    d_tokens = set(tokenize(text))
    score = 0.0
    for token in q_tokens:
        if token in d_tokens:
            score += 1.0
            if any(ch.isdigit() for ch in token):
                score += 1.5
    return score


def retrieve_top1(query):
    scored = []
    for doc_id, text in enumerate(kb):
        scored.append((lexical_score(query, text), doc_id, text))
    scored.sort(reverse=True)
    best_score, best_doc_id, best_text = scored[0]
    return {
        'score': best_score,
        'doc_id': best_doc_id,
        'context': best_text,
    }


def toy_generator(query, context):
    q = query.lower()
    if 'e401' in q:
        if 'e401' in context.lower():
            return 'E401 means authentication failed because the token is invalid or expired.'
        return 'E401 is a network timeout error.'

    if 'password' in q:
        if 'reset password' in context.lower():
            return 'Open Settings, go to Security, and choose Reset Password.'
        return 'You need to contact support to change your password.'

    if 'chunking' in q:
        if 'chunking' in context.lower():
            return 'Chunking breaks long documents into smaller passages so retrieval becomes more focused.'
        return 'Chunking is used to compress embeddings.'

    if 'embedding' in q:
        if 'embedding' in context.lower():
            return 'Embeddings turn text into vectors so semantic similarity can be measured.'
        return 'Embeddings are only used to store passwords securely.'

    return 'I do not know.'


def rag_answer(query):
    retrieval = retrieve_top1(query)
    answer = toy_generator(query, retrieval['context'])
    return {
        'query': query,
        'doc_id': retrieval['doc_id'],
        'context': retrieval['context'],
        'answer': answer,
    }

## 3. 先看看单条样本

我们先看一条 query 的完整过程：
- 找到了哪个上下文
- 最后生成了什么答案

In [ ]:
example = dataset[0]
result = rag_answer(example['query'])
print('query:')
print(result['query'])
print('\nretrieved doc_id:', result['doc_id'])
print('context:')
print(result['context'])
print('\nanswer:')
print(result['answer'])
print('\nreference_answer:')
print(example['reference_answer'])

## 4. 生成阶段常看哪些维度

在真实 RAG 里，生成答案常见会看这些维度：

- `Correctness`：答案是否正确
- `Faithfulness / Groundedness`：答案是否真的被上下文支持
- `Relevance`：答案是否真正回答了问题
- `Completeness`：答案是否缺关键点

这一课我们先做一个教学版近似：

- `Exact Match`：和参考答案是否完全一致
- `Token F1`：和参考答案在词面上重合程度如何
- `Context Support`：答案里的关键信息是否能在上下文里找到

这些都不是工业级标准，但很适合入门。

In [ ]:
def normalize_text(text):
    tokens = tokenize(text)
    return ' '.join(tokens)


def exact_match(prediction, reference):
    return 1.0 if normalize_text(prediction) == normalize_text(reference) else 0.0


def token_f1(prediction, reference):
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)
    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    overlap = sum((pred_counter & ref_counter).values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def context_support_score(answer, context):
    answer_tokens = [t for t in tokenize(answer) if len(t) > 3]
    context_tokens = set(tokenize(context))
    if not answer_tokens:
        return 0.0
    covered = sum(1 for token in answer_tokens if token in context_tokens)
    return covered / len(answer_tokens)

## 5. 手动看一条答案的分数

这样你会更直观地理解这些指标到底在比较什么。

In [ ]:
item = dataset[1]
pred = rag_answer(item['query'])

print('query:', item['query'])
print('retrieved doc_id:', pred['doc_id'])
print('answer:', pred['answer'])
print('reference:', item['reference_answer'])
print('context:', pred['context'])
print()
print('Exact Match   =', exact_match(pred['answer'], item['reference_answer']))
print('Token F1      =', round(token_f1(pred['answer'], item['reference_answer']), 3))
print('ContextSupport=', round(context_support_score(pred['answer'], pred['context']), 3))

## 6. 批量评估整个 RAG 系统

注意这里评估的已经不是“检索器单独表现”，而是：

**检索 + 生成 合在一起后的最终表现。**

In [ ]:
def evaluate_rag_generation(dataset):
    rows = []
    em_scores = []
    f1_scores = []
    support_scores = []
    retrieval_hit_scores = []

    for item in dataset:
        pred = rag_answer(item['query'])
        retrieval_hit = 1.0 if pred['doc_id'] == item['gold_doc_id'] else 0.0
        em = exact_match(pred['answer'], item['reference_answer'])
        f1 = token_f1(pred['answer'], item['reference_answer'])
        support = context_support_score(pred['answer'], pred['context'])

        retrieval_hit_scores.append(retrieval_hit)
        em_scores.append(em)
        f1_scores.append(f1)
        support_scores.append(support)

        rows.append({
            'query': item['query'],
            'gold_doc_id': item['gold_doc_id'],
            'retrieved_doc_id': pred['doc_id'],
            'retrieval_hit': retrieval_hit,
            'answer': pred['answer'],
            'reference': item['reference_answer'],
            'exact_match': em,
            'token_f1': f1,
            'context_support': support,
        })

    summary = {
        'RetrievalHit@1': sum(retrieval_hit_scores) / len(retrieval_hit_scores),
        'ExactMatch': sum(em_scores) / len(em_scores),
        'TokenF1': sum(f1_scores) / len(f1_scores),
        'ContextSupport': sum(support_scores) / len(support_scores),
    }
    return summary, rows

In [ ]:
summary, rows = evaluate_rag_generation(dataset)
print(summary)

## 7. 逐条分析结果

这是最接近真实工作流的一步。
你会发现：很多时候不是一个总分就能说明问题，而是要看失败样本到底怎么失败。

In [ ]:
for row in rows:
    print('query:', row['query'])
    print('gold_doc_id:', row['gold_doc_id'], '| retrieved_doc_id:', row['retrieved_doc_id'], '| retrieval_hit:', row['retrieval_hit'])
    print('answer:', row['answer'])
    print('reference:', row['reference'])
    print('exact_match:', row['exact_match'], '| token_f1:', round(row['token_f1'], 3), '| context_support:', round(row['context_support'], 3))
    print('-' * 90)

## 8. 一个很重要的观察

你现在应该能看到一个关键事实：

**检索评估好，不代表生成评估一定也好。**

原因可能有很多：
- 上下文虽然相关，但模型没用好
- 上下文里有答案，但模型表述错了
- 模型混入了额外幻觉内容
- 上下文只覆盖了部分信息，答案却写得太满

所以真实 RAG 系统里，经常会把评估拆成两层：

1. `Retrieval Evaluation`
2. `Generation Evaluation`

这样你才能定位问题到底出在哪一层。

## 9. 本课小结

这节课你要带走的核心是：

1. RAG 的评估不只有检索指标，还要看答案质量。
2. `Exact Match / Token F1 / Context Support` 可以作为很好的入门近似。
3. 真正调系统时，一定要把“检索问题”和“生成问题”拆开看。

下一课最自然的衔接就是：
**如何构建一个更像真实项目的 RAG eval set**，也就是数据集设计与错误分析。